In [0]:
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.functions import *

from dataclasses import dataclass

In [0]:
# %sql
# create or replace table session_11_framework.config_table (
#     pipeline_name string, 
#     file_path string, 
#     schema_details map<string,string>, 
#     keys array<string>,
#     table_name string,
#     encrypt_col array<string>,
#     write_mode string,
#     source_option map<string, string>
# )

In [0]:
def upsert_into(df:DataFrame, table_name:str, keys:list) -> DataFrame:
    delta_obj = DeltaTable.forName(spark,table_name)
    return (
        delta_obj.alias("t").merge(
            df.alias("s")," AND ".join([f"t.{key} = s.{key}" for key in keys])
            )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
        )

In [0]:
data = [{
    "pipeline_name": "book_lib",
    "file_path": "/Workspace/Repos/projects/databricks-batch-pipeline/data/books2.csv",
    "table_name": "book_lib",
    "schema_details": {
        "isbn13": "string",
        "title": "string",
        "subtitle": "string",
        "authors": "string",
        "categories": "string",
        "published_year": "int",
        "average_rating": "double",
        "num_pages": "int",
        "ratings_count": "int"
    },
    "source_option": {
        "header": "true",
        "delimiter": ",",
        "multiline": "true",
        "escape_option": "\"",
        "quote_option": "\"",
    },
    "keys": ["isbn13"],
    "write_mode": "scd2",
    "encrypt_col": []
    }
]

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("schema_details", MapType(StringType(), StringType()), True),
    StructField("keys", ArrayType(StringType()), True),
    StructField("write_mode", StringType(), True),
    StructField("source_option", MapType(StringType(), StringType()), True),
    StructField("encrypt_col", ArrayType(StringType()), True)
])
mock_df = spark.createDataFrame(data, schema)
# mock_df.display()
upsert_into(mock_df, "session_11_framework.config_table", ["pipeline_name"])

In [0]:
data = [{
    "pipeline_name": "hr",
    "file_path": "/Workspace/Repos/projects/databricks-batch-pipeline/data/HRDataset_v14.csv",
    "table_name": "hr",
    "schema_details": {
        "Employee_name": "string",
        "EmpID": "string",
        "Salary": "string",
        "HispanicLatino": "string",
        "EngagementSurvey": "double",
        "EmpSatisfaction": "int",
    },
    "source_option": {
        "header": "true",
        "delimiter": ",",
        "multiline": "true",
        "escape_option": "\"",
        "quote_option": "\"",
    },
    "keys": ["EmpID"],
    "write_mode": "scd2",
    "encrypt_col": ["Employee_name", "Salary", "ManagerID"]
    }
]

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("schema_details", MapType(StringType(), StringType()), True),
    StructField("keys", ArrayType(StringType()), True),
    StructField("write_mode", StringType(), True),
    StructField("source_option", MapType(StringType(), StringType()), True),
    StructField("encrypt_col", ArrayType(StringType()), True)
])
mock_df = spark.createDataFrame(data, schema)
# mock_df.display()
upsert_into(mock_df, "session_11_framework.config_table", ["pipeline_name"])

In [0]:
%sql
select * from workspace.default.hr_silver